In [4]:
%%writefile app.py
import streamlit as st
import io
import os
from PyPDF2 import PdfReader
from fpdf import FPDF
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from google.colab import userdata

# --- CLASS DEFINITION (The Logic) ---
class MCQGenerator:
    def __init__(self, api_key):
        # Using Llama 3.3 via Groq for high-speed generation
        self.llm = ChatGroq(
            temperature=0.3,
            model_name="llama-3.3-70b-versatile",
            groq_api_key=api_key
        )

    def extract_text(self, pdf_file):
        """Extracts text from the uploaded PDF file object."""
        reader = PdfReader(pdf_file)
        text = ""
        for page in reader.pages:
            content = page.extract_text()
            if content:
                text += content + "\n"
        return text

    def generate_quiz(self, context):
        """Sends extracted text to Groq to generate 5 MCQs."""
        template = """
        You are an expert educator. Based on the following text, create 5 Multiple Choice Questions (MCQs).

        Format exactly as follows:
        Question [Number]: [Question text]
        A) [Option 1]
        B) [Option 2]
        C) [Option 3]
        D) [Option 4]
        Correct Answer: [Letter]

        TEXT:
        {context}
        """
        prompt = PromptTemplate.from_template(template)
        # Modern LCEL Chain (The Pipe Operator)
        chain = prompt | self.llm
        response = chain.invoke({"context": context})
        return response.content

    def create_pdf_bytes(self, mcq_text):
        """Converts the MCQ string into a downloadable PDF byte stream."""
        pdf = FPDF()
        pdf.add_page()
        pdf.set_font("Arial", size=12)
        # Sanitize text for Latin-1 encoding (FPDF default)
        safe_text = mcq_text.encode('latin-1', 'replace').decode('latin-1')
        for line in safe_text.split('\n'):
            pdf.multi_cell(0, 10, txt=line)
        return pdf.output(dest='S').encode('latin-1')

# --- STREAMLIT UI (The Interface) ---
st.set_page_config(page_title="AI MCQ Creator", page_icon="🎓")
st.title("📄 PDF to MCQ Generator")
st.markdown("---")

# 1. Initialize Session State (Persistent Memory)
if "quiz_text" not in st.session_state:
    st.session_state.quiz_text = None

# 2. Initialize Logic Class
try:
    # Look for the key in the system environment variables instead
    key = os.environ.get('GROQ_API_KEY')
    if not key:
        st.error("GROQ_API_KEY not found in environment.")
    generator = MCQGenerator(key)
except Exception as e:
    st.error(f"Initialization Error: {e}")

# 3. File Upload UI
uploaded_file = st.file_uploader("Upload your lecture or notes (PDF)", type="pdf")

if uploaded_file:
    # Button to trigger the backend pipeline
    if st.button("Generate Quiz"):
        with st.spinner("Grok is analyzing your document and writing questions..."):
            # Step A: Extract
            raw_text = generator.extract_text(uploaded_file)
            # Step B: Generate
            quiz_result = generator.generate_quiz(raw_text)
            # Step C: Save to Memory
            st.session_state.quiz_text = quiz_result

# 4. Results Display & Download
if st.session_state.quiz_text:
    st.success("Quiz Generated Successfully!")
    st.subheader("Quiz Preview")
    st.text(st.session_state.quiz_text)

    # Generate PDF bytes only when needed
    pdf_data = generator.create_pdf_bytes(st.session_state.quiz_text)

    st.download_button(
        label="📥 Download MCQ as PDF",
        data=pdf_data,
        file_name="grok_generated_quiz.pdf",
        mime="application/pdf"
    )

Overwriting app.py


In [6]:
import os
from google.colab import userdata

# Set the key in the notebook environment
os.environ["GROQ_API_KEY"] = userdata.get('RealEst')

import urllib
print("Password/IP:", urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
!streamlit run app.py & npx localtunnel --port 8501

Password/IP: 35.237.105.160
⠙⠹

⠸⠼⠴⠦⠧your url is: https://busy-bottles-look.loca.lt

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.237.105.160:8501

  Stopping...
  Stopping...
^C
